In [41]:
from langchain_cohere import ChatCohere,CohereEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from dotenv import load_dotenv
from  langchain_huggingface import HuggingFaceEmbeddings

In [42]:
loader = PyPDFLoader('Deep Learning by Ian Goodfellow, Yoshua Bengio, Aaron Courville.pdf')
docs = loader.load()

In [43]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
)

In [44]:
chunks = splitter.split_documents(docs)
len(chunks)

2425

In [45]:
chunks[1]

Document(metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2019-01-26T22:50:33+16:00', 'enhanced': 'By PDF Enhancer 3.5.6412/Unix', 'spdf': '1127', 'source': 'Deep Learning by Ian Goodfellow, Yoshua Bengio, Aaron Courville.pdf', 'total_pages': 801, 'page': 2, 'page_label': '3'}, page_content='Contents\nWebsite vii\nAcknowledgments viii\nNotation xi\n1 Introduction 1\n1.1 Who Should Read This Book? . . . . . . . . . . . . . . . . . . . . 8\n1.2 Historical Trends in Deep Learning . . . . . . . . . . . . . . . . . 11\nI Applied Math and Machine Learning Basics 29\n2 Linear Algebra 31\n2.1 Scalars, Vectors, Matrices and Tensors . . . . . . . . . . . . . . . 31\n2.2 Multiplying Matrices and Vectors . . . . . . . . . . . . . . . . . . 34\n2.3 Identity and Inverse Matrices . . . . . . . . . . . . . . . . . . . . 36\n2.4 Linear Dependence and Span . . . . . . . . . . . . . . . . . . . . 37\n2.5 Norms . . . . . . . . . . . . . . . . . . . . . . . 

In [46]:
# Embedding Generation and Storing in vector store
load_dotenv()
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks,embeddings)

In [47]:
vector_store.index_to_docstore_id

{0: 'de4dc2ec-78fd-46fd-8d01-eeb4dbb0a830',
 1: '8d075ea9-60c5-4115-b6ff-697adfd9a39b',
 2: '293a0488-bb9b-4a5e-8417-5d60e1e48604',
 3: '1397f6d4-5d96-4dad-a73b-b2d03a6f5716',
 4: 'd698cbce-dc19-4d25-9327-165ef93162e4',
 5: '752f551b-b132-41b9-9b4a-961fadcf27f2',
 6: 'bfd6a9d7-cddb-4db2-ba9d-948c57c93b67',
 7: '13858961-ea05-447d-91c9-29da172795c7',
 8: 'ed779b81-f941-4050-b2ad-f3fe16e0253f',
 9: '90b968f7-5959-4386-95e0-3793beec75f4',
 10: '0645dc17-22fc-4029-bbb4-5aaeea6778e2',
 11: '7b557098-b59d-455e-aed9-22cfa68bcc2c',
 12: 'e9015170-06ce-4e41-9a03-cb4780ff4cf4',
 13: '2facd8e9-0726-46a3-8587-59cbaa8a30e5',
 14: 'c28cf826-8432-4e9c-8036-a2d588fd1647',
 15: 'd23a075e-b6aa-4952-ad1c-6692385e1f10',
 16: 'b90d9699-9dba-4dbf-adb2-c16572a586fc',
 17: '97c73f08-5a81-4319-bb82-1dcd7a75fa9e',
 18: '13ba6f30-4866-4141-af4f-6bf2a29832b8',
 19: 'c92c1ec5-3bec-407c-a8a4-ea2fddc8d812',
 20: '966499d2-da79-403a-a784-c26b510fffb5',
 21: '81bca304-6a84-4c96-bd6f-302b94c539ae',
 22: '6812fb34-f805-

In [48]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})
def format_doc(retrieved_Doc):
    context_text="\n\n".join(doc.page_content for doc in retrieved_Doc)
    return context_text

In [49]:
llm = ChatCohere(model="command-a-03-2025")

In [50]:
prompt = PromptTemplate(
    template="""
    You are helpful assistance.
    If the context is insufficient, then say me , I don't know.

    {context}
    Question = {question}
    """,
    input_variables=["context","question"]
)

In [51]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_doc),
    "question": RunnablePassthrough()
})

In [52]:
parser = StrOutputParser()

In [53]:
main_chain = parallel_chain | prompt | llm | parser

In [54]:
main_chain.invoke("what is exact meaning of DeepLearning?")

'The term "Deep Learning" refers to a subfield of machine learning that focuses on the use of **deep neural networks**—neural networks with many layers (hence "deep"). The key idea is that these networks can learn hierarchical representations of data, where each layer captures increasingly complex features or patterns from the input data. For example, in image recognition, early layers might detect edges, while deeper layers recognize more abstract features like shapes or objects.\n\nThe term gained popularity in the mid-2000s, particularly after **Geoffrey Hinton** and others demonstrated that deep neural networks could be efficiently trained using techniques like **greedy layer-wise pre-training**. This breakthrough allowed researchers to train deeper networks than previously possible, leading to significant improvements in performance on tasks like image and speech recognition.\n\nIn essence, "Deep Learning" emphasizes both the **depth** of the network architecture (the number of la